In [ ]:
# =======================================================================
#                    Change Detection using U-Net
# =======================================================================
#
#   Dataset  : LEVIR-CD  (via Google Drive zip)
#   ZIP_PATH : /content/drive/MyDrive/Change Detection/dataset.zip
#
#   Dataset layout expected:
#     dataset/
#         train/  A/  B/  label/
#         val/    A/  B/  label/
#         test/   A/  B/  label/
#
#   Patch strategy:
#     • No resizing — images stay at native 1024×1024
#     • Each image → 4×4 grid of 256×256 patches (16 total)
#     • Each image is loaded from disk ONCE per worker (LRU cache)
# =======================================================================

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
import time
import random
import zipfile
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

In [ ]:
# CONFIG
class CFG:
    # ---- paths ---------------------------------------------------------------
    ZIP_PATH    = "/content/drive/MyDrive/Change Detection/dataset.zip"
    EXTRACT_DIR = "/content"
    DATA_ROOT   = "/content/dataset"
    SAVE_DIR    = "/content/drive/MyDrive/Change Detection"
    MODEL_PATH  = f"{SAVE_DIR}/best_model.pth"

    # ---- patch strategy (no resize) ------------------------------------------
    # Original LEVIR-CD images are 1024×1024.
    # We tile each image into a 4×4 grid of 256×256 non-overlapping patches.
    IMG_SIZE   = 1024                        # native resolution — never resized
    PATCH_SIZE = 256                         # each patch is 256×256
    GRID       = IMG_SIZE // PATCH_SIZE      # 4  (grid steps per axis)
    N_PATCHES  = GRID ** 2                   # 16 patches per image

    # ---- caching -------------------------------------------------------------
    # Max number of full image triplets (A + B + label) held in RAM per worker.
    # One triplet ≈ 3 tensors × 1024 × 1024 × 4 bytes ≈ 12 MB.
    # 64 entries ≈ 768 MB — well within a Colab T4 session (15 GB RAM).
    # Set to None to cache the entire split (no eviction).
    CACHE_SIZE = 64

    # ---- training ------------------------------------------------------------
    EPOCHS      = 15
    BATCH_SIZE  = 4
    LR          = 1e-4
    NUM_WORKERS = 2
    PIN_MEMORY  = True

    # ---- inference -----------------------------------------------------------
    THRESHOLD = 0.4

    # ---- device --------------------------------------------------------------
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(CFG.SAVE_DIR, exist_ok=True)
print(f"[INFO] Device     : {CFG.DEVICE}")
print(f"[INFO] Patch size : {CFG.PATCH_SIZE}×{CFG.PATCH_SIZE}")
print(f"[INFO] Grid       : {CFG.GRID}×{CFG.GRID}  ({CFG.N_PATCHES} patches / image)")
print(f"[INFO] Cache size : {CFG.CACHE_SIZE} triplets / worker")

[INFO] Device     : cuda
[INFO] Patch size : 256×256
[INFO] Grid       : 4×4  (16 patches / image)
[INFO] Cache size : 64 triplets / worker


In [ ]:
# DATASET SETUP
def setup_dataset():

    dataset_dir = Path("/content/LEVIR CD")
    local_zip   = Path("/content/dataset.zip")

    # 1. Dataset already extracted
    if dataset_dir.exists():
        CFG.DATA_ROOT = str(dataset_dir)
        print(f"✅ Dataset already exists → {CFG.DATA_ROOT}")
        return

    # 2. Verify Google Drive zip exists
    if not Path(CFG.ZIP_PATH).exists():
        raise FileNotFoundError(
            f"Dataset zip not found:\n{CFG.ZIP_PATH}"
        )

    # 3. Copy zip from Drive → local Colab storage
    print("Copying dataset.zip to local storage...")

    os.system(
        f'cp "{CFG.ZIP_PATH}" "/content/dataset.zip"'
    )

    if not local_zip.exists():
        raise RuntimeError("Failed to copy dataset.zip")

    # 4. Extract locally
    print("Extracting dataset locally...")

    try:
        with zipfile.ZipFile(local_zip, "r") as zf:
            zf.extractall("/content")

    except Exception as e:
        raise RuntimeError(
            f"Dataset extraction failed:\n{e}"
        )

    # 5. Verify extraction
    if not dataset_dir.exists():
        raise RuntimeError(
            "Extraction finished but '/content/LEVIR CD' not found."
        )

    CFG.DATA_ROOT = str(dataset_dir)

    print(f"✅ Dataset extracted → {CFG.DATA_ROOT}")

In [ ]:
setup_dataset()

Copying dataset.zip to local storage...
Extracting dataset locally...
✅ Dataset extracted → /content/LEVIR CD


In [ ]:
root = Path("/content/LEVIR CD")

print("\nTRAIN")
print("A:", len(list((root / "train" / "A").glob("*"))))
print("B:", len(list((root / "train" / "B").glob("*"))))
print("label:", len(list((root / "train" / "label").glob("*"))))

print("\nVAL")
print("A:", len(list((root / "val" / "A").glob("*"))))
print("B:", len(list((root / "val" / "B").glob("*"))))
print("label:", len(list((root / "val" / "label").glob("*"))))

print("\nTEST")
print("A:", len(list((root / "test" / "A").glob("*"))))
print("B:", len(list((root / "test" / "B").glob("*"))))
print("label:", len(list((root / "test" / "label").glob("*"))))


TRAIN
A: 445
B: 445
label: 445

VAL
A: 64
B: 64
label: 64

TEST
A: 128
B: 128
label: 128


In [ ]:
# LRU IMAGE CACHE
class _ImageCache:
    """
    Bounded LRU cache for full-resolution image triplets.

    Stores pre-processed float32 tensors (A, B, label) keyed by image index.
    Each entry is populated on first access and evicted least-recently-used
    when the cache reaches `maxsize` entries.

    Why a standalone class instead of lru_cache on a method?
    ─────────────────────────────────────────────────────────
    functools.lru_cache cannot decorate bound instance methods because
    `self` is part of the cache key and Dataset objects are not hashable.
    Wrapping the loader in a plain callable object sidesteps that restriction.

    Why per-worker instead of shared?
    ───────────────────────────────────
    PyTorch DataLoader forks each worker as a separate subprocess. Shared
    memory across processes requires explicit IPC (e.g. torch.multiprocessing
    shared tensors), which adds complexity and is only worthwhile for very
    large datasets. Per-worker caches are simpler, lock-free, and already
    eliminate the dominant cost (16 disk reads → 1 per image per worker).
    """

    def __init__(
        self,
        a_paths:     list,
        b_paths:     list,
        label_paths: list,
        img_tf,
        lbl_tf,
        maxsize: int | None = 64,
    ):
        self._a_paths     = a_paths
        self._b_paths     = b_paths
        self._label_paths = label_paths
        self._img_tf      = img_tf
        self._lbl_tf      = lbl_tf

        # Bind lru_cache to the internal loader at construction time so the
        # cache is tied to this specific _ImageCache instance.
        self._cached_load = lru_cache(maxsize=maxsize)(self._load)

    # ── internal loader — called at most once per (worker, img_idx) ───────────

    def _load(self, img_idx: int):
        """
        Open one image triplet from disk and convert to float32 tensors.
        The return value is a plain tuple so it is safe to cache.
        Labels are binarised here — downstream code never receives raw grays.
        """
        img_a = self._img_tf(
            Image.open(self._a_paths[img_idx]).convert("RGB")
        )                                           # 3 × 1024 × 1024  float32
        img_b = self._img_tf(
            Image.open(self._b_paths[img_idx]).convert("RGB")
        )                                           # 3 × 1024 × 1024  float32
        label = self._lbl_tf(
            Image.open(self._label_paths[img_idx]).convert("L")
        )                                           # 1 × 1024 × 1024  float32
        label = (label > 0.5).float()               # hard binary {0.0, 1.0}

        return img_a, img_b, label                  # tuple is LRU-cacheable

    # ── public accessor ───────────────────────────────────────────────────────

    def get(self, img_idx: int):
        """Return (img_a, img_b, label) tensors for the given image index."""
        return self._cached_load(img_idx)

    # ── diagnostics ───────────────────────────────────────────────────────────

    def info(self) -> str:
        """Human-readable hit / miss summary."""
        ci = self._cached_load.cache_info()
        total    = ci.hits + ci.misses
        hit_rate = (ci.hits / total * 100) if total > 0 else 0.0
        return (
            f"LRU cache — hits: {ci.hits} | misses: {ci.misses} | "
            f"hit-rate: {hit_rate:.1f}% | "
            f"size: {ci.currsize} / {ci.maxsize}"
        )

    def clear(self):
        """Evict all cached entries."""
        self._cached_load.cache_clear()

In [ ]:
# DATASET
class ChangeDetectionDataset(Dataset):
    """
    Yields (x, label_patch) pairs for change detection.

    x           : Float32 tensor  6 × 256 × 256  (A and B patches concatenated)
    label_patch : Float32 tensor  1 × 256 × 256  (binary change mask)

    Patch indexing
    ──────────────
    global_idx = image_idx × N_PATCHES + patch_idx
    patch_idx  = row × GRID + col        (row, col ∈ {0, 1, 2, 3})

    Caching
    ───────
    Each full 1024×1024 image triplet is loaded from disk exactly once
    per worker process. All 16 patch requests for the same image are
    served from the in-process LRU cache on accesses 2–16, reducing
    disk I/O by up to 15× compared with the naive implementation.
    """

    def __init__(self, split: str):
        root = Path(CFG.DATA_ROOT) / split

        self.a_paths     = sorted((root / "A").glob("*"))
        self.b_paths     = sorted((root / "B").glob("*"))
        self.label_paths = sorted((root / "label").glob("*"))

        assert (
            len(self.a_paths)
            == len(self.b_paths)
            == len(self.label_paths)
        ), (
            f"[{split}] File count mismatch — "
            f"A={len(self.a_paths)}, "
            f"B={len(self.b_paths)}, "
            f"label={len(self.label_paths)}"
        )

        self.n_images  = len(self.a_paths)
        self.n_patches = CFG.N_PATCHES    # 16
        self.grid      = CFG.GRID         # 4

        # ToTensor only — NO Resize.
        # PIL uint8 [0, 255] → float32 [0.0, 1.0], shape C×H×W.
        img_tf = transforms.ToTensor()
        lbl_tf = transforms.ToTensor()

        # One LRU cache per Dataset instance.
        # When DataLoader forks workers, each subprocess receives its own
        # copy of this object (and therefore its own isolated cache).
        self._cache = _ImageCache(
            a_paths     = self.a_paths,
            b_paths     = self.b_paths,
            label_paths = self.label_paths,
            img_tf      = img_tf,
            lbl_tf      = lbl_tf,
            maxsize     = CFG.CACHE_SIZE,
        )

    # ── length ────────────────────────────────────────────────────────────────

    def __len__(self) -> int:
        return self.n_images * self.n_patches

    # ── patch crop helper ─────────────────────────────────────────────────────

    @staticmethod
    def _crop(tensor: torch.Tensor, row: int, col: int) -> torch.Tensor:
        """
        Slice a CFG.PATCH_SIZE × CFG.PATCH_SIZE window from a C×H×W tensor.
        Pure index arithmetic — no data is copied until the DataLoader
        collates the batch.
        """
        p = CFG.PATCH_SIZE
        return tensor[
            :,
            row * p : (row + 1) * p,
            col * p : (col + 1) * p,
        ]

    # ── main accessor ─────────────────────────────────────────────────────────

    def __getitem__(self, idx: int):
        # ── 1. Decode flat index → (image, row, col) ──────────────────────
        img_idx   = idx // self.n_patches    # 0 … n_images - 1
        patch_idx = idx  % self.n_patches    # 0 … 15
        row       = patch_idx // self.grid   # 0 … 3
        col       = patch_idx  % self.grid   # 0 … 3

        # ── 2. Retrieve full-resolution triplet (1 disk read per image) ───
        img_a, img_b, label = self._cache.get(img_idx)

        # ── 3. Crop spatially aligned 256×256 patches ─────────────────────
        patch_a = self._crop(img_a,  row, col)   # 3 × 256 × 256
        patch_b = self._crop(img_b,  row, col)   # 3 × 256 × 256
        patch_l = self._crop(label,  row, col)   # 1 × 256 × 256

        # ── 4. Build 6-channel input (A ∥ B) ─────────────────────────────
        x = torch.cat([patch_a, patch_b], dim=0)   # 6 × 256 × 256

        return x, patch_l

    # ── diagnostics ───────────────────────────────────────────────────────────

    def cache_info(self) -> str:
        return self._cache.info()

In [ ]:
# DATALOADERS
def build_loaders():
    train_ds = ChangeDetectionDataset("train")
    val_ds   = ChangeDetectionDataset("val")
    test_ds  = ChangeDetectionDataset("test")

    peak_mb = (CFG.CACHE_SIZE or train_ds.n_images) * 12
    print(
        f"[INFO] Images  — "
        f"train: {train_ds.n_images} | "
        f"val: {val_ds.n_images} | "
        f"test: {test_ds.n_images}"
    )
    print(
        f"[INFO] Patches — "
        f"train: {len(train_ds)} | "
        f"val: {len(val_ds)} | "
        f"test: {len(test_ds)}"
    )
    print(
        f"[INFO] Cache   — "
        f"maxsize: {CFG.CACHE_SIZE} triplets | "
        f"≈ {peak_mb} MB peak RAM per worker"
    )

    train_loader = DataLoader(
        train_ds,
        batch_size  = CFG.BATCH_SIZE,
        shuffle     = True,
        num_workers = CFG.NUM_WORKERS,
        pin_memory  = CFG.PIN_MEMORY,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size  = CFG.BATCH_SIZE,
        shuffle     = False,
        num_workers = CFG.NUM_WORKERS,
        pin_memory  = CFG.PIN_MEMORY,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size  = CFG.BATCH_SIZE,
        shuffle     = False,
        num_workers = CFG.NUM_WORKERS,
        pin_memory  = CFG.PIN_MEMORY,
    )
    return train_loader, val_loader, test_loader

In [ ]:
# U-NET ARCHITECTURE
class ConvBlock(nn.Module):
    """Two consecutive Conv2d → BatchNorm → ReLU layers."""

    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class EncoderBlock(nn.Module):
    """ConvBlock → MaxPool. Returns (downsampled, skip)."""

    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.conv = ConvBlock(in_ch, out_ch)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x: torch.Tensor):
        skip = self.conv(x)
        return self.pool(skip), skip


class DecoderBlock(nn.Module):
    """ConvTranspose2d upsample → concat skip → ConvBlock."""

    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = ConvBlock(out_ch * 2, out_ch)   # ×2 due to skip concat

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)
        # Pad to handle odd spatial dimensions gracefully
        if x.shape != skip.shape:
            x = F.pad(
                x,
                [0, skip.shape[3] - x.shape[3],
                 0, skip.shape[2] - x.shape[2]],
            )
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)


class UNet(nn.Module):
    """
    Standard U-Net for binary change detection.
      Input  : B × 6 × H × W   (A and B patches concatenated channel-wise)
      Output : B × 1 × H × W   (change probability map in [0, 1])
    """

    def __init__(
        self,
        in_ch: int = 6,
        out_ch: int = 1,
        features: tuple = (64, 128, 256, 512),
    ):
        super().__init__()

        # Encoder
        self.enc1 = EncoderBlock(in_ch,        features[0])
        self.enc2 = EncoderBlock(features[0],  features[1])
        self.enc3 = EncoderBlock(features[1],  features[2])
        self.enc4 = EncoderBlock(features[2],  features[3])

        # Bottleneck
        self.bottleneck = ConvBlock(features[3], features[3] * 2)

        # Decoder
        self.dec4 = DecoderBlock(features[3] * 2, features[3])
        self.dec3 = DecoderBlock(features[3],     features[2])
        self.dec2 = DecoderBlock(features[2],     features[1])
        self.dec1 = DecoderBlock(features[1],     features[0])

        # Output head
        self.head = nn.Sequential(
            nn.Conv2d(features[0], out_ch, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x, s1 = self.enc1(x)
        x, s2 = self.enc2(x)
        x, s3 = self.enc3(x)
        x, s4 = self.enc4(x)

        x = self.bottleneck(x)

        x = self.dec4(x, s4)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x, s1)

        return self.head(x)

In [ ]:
# LOSS FUNCTION
class BCEDiceLoss(nn.Module):
    """Combined loss: 0.5 × Binary Cross-Entropy + 0.5 × Dice."""

    def __init__(self, smooth: float = 1e-6):
        super().__init__()
        self.bce    = nn.BCELoss()
        self.smooth = smooth

    def dice_loss(
        self, pred: torch.Tensor, target: torch.Tensor
    ) -> torch.Tensor:
        pred   = pred.view(-1)
        target = target.view(-1)
        inter  = (pred * target).sum()
        dice   = (2.0 * inter + self.smooth) / (
                  pred.sum() + target.sum() + self.smooth)
        return 1.0 - dice

    def forward(
        self, pred: torch.Tensor, target: torch.Tensor
    ) -> torch.Tensor:
        return 0.5 * self.bce(pred, target) + 0.5 * self.dice_loss(pred, target)

In [ ]:
# METRICS
def compute_metrics(
    preds: torch.Tensor,
    targets: torch.Tensor,
    threshold: float = 0.5,
) -> dict:
    """
    Compute Precision, Recall, F1 Score, and IoU from probability maps.
    Inputs may be on any device; binarisation is applied internally.
    """
    preds   = (preds > threshold).float().view(-1)
    targets = targets.float().view(-1)

    tp = (preds * targets).sum().item()
    fp = (preds * (1 - targets)).sum().item()
    fn = ((1 - preds) * targets).sum().item()

    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    iou       = tp / (tp + fp + fn + 1e-8)

    return {"precision": precision, "recall": recall, "f1": f1, "iou": iou}

In [ ]:
# TRAINING FUNCTIONS
def train_one_epoch(model, loader, optimizer, criterion, device) -> float:
    model.train()
    total_loss = 0.0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(
    model, loader, criterion, device, threshold: float = 0.5
) -> dict:
    model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        pred = model(x)
        loss = criterion(pred, y)
        total_loss += loss.item() * x.size(0)

        all_preds.append(pred.cpu())
        all_targets.append(y.cpu())

    all_preds   = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)

    metrics         = compute_metrics(all_preds, all_targets, threshold)
    metrics["loss"] = total_loss / len(loader.dataset)
    return metrics


def train(model, train_loader, val_loader, device) -> list:
    criterion = BCEDiceLoss().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG.LR)

    best_iou = 0.0
    history  = []

    print(
        f"\n{'Epoch':>6} │ {'Train Loss':>11} │ {'Val Loss':>9} │ "
        f"{'Val IoU':>8} │ {'Val F1':>7} │ {'Time':>6}"
    )
    print("─" * 62)

    for epoch in range(1, CFG.EPOCHS + 1):
        t0 = time.time()

        train_loss  = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )
        val_metrics = evaluate(
            model, val_loader, criterion, device, threshold=0.5
        )
        elapsed = time.time() - t0

        print(
            f"{epoch:>6} │ {train_loss:>11.5f} │ "
            f"{val_metrics['loss']:>9.5f} │ "
            f"{val_metrics['iou']:>8.4f} │ "
            f"{val_metrics['f1']:>7.4f} │ "
            f"{elapsed:>5.1f}s"
        )

        history.append(
            {"epoch": epoch, "train_loss": train_loss, **val_metrics}
        )

        if val_metrics["iou"] > best_iou:
            best_iou = val_metrics["iou"]
            torch.save(model.state_dict(), CFG.MODEL_PATH)
            print(f"         ✔ Best model saved (IoU = {best_iou:.4f})")

    print(f"\n[INFO] Training complete.  Best Val IoU : {best_iou:.4f}")
    print(f"[INFO] Best model saved  → {CFG.MODEL_PATH}")
    return history

In [ ]:
# TESTING
@torch.no_grad()
def test(model, test_loader, device) -> dict:
    print("\n[INFO] Loading best model weights for testing …")
    model.load_state_dict(
        torch.load(CFG.MODEL_PATH, map_location=device)
    )
    model.eval()

    criterion  = BCEDiceLoss().to(device)
    total_loss = 0.0
    all_preds, all_targets = [], []

    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        loss = criterion(pred, y)
        total_loss += loss.item() * x.size(0)
        all_preds.append(pred.cpu())
        all_targets.append(y.cpu())

    all_preds   = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)

    metrics         = compute_metrics(
        all_preds, all_targets, threshold=CFG.THRESHOLD
    )
    metrics["loss"] = total_loss / len(test_loader.dataset)

    print("\n" + "═" * 42)
    print("           TEST RESULTS")
    print("═" * 42)
    print(f"  Threshold  : {CFG.THRESHOLD}")
    print(f"  Test Loss  : {metrics['loss']:.5f}")
    print(f"  Precision  : {metrics['precision']:.4f}")
    print(f"  Recall     : {metrics['recall']:.4f}")
    print(f"  F1 Score   : {metrics['f1']:.4f}")
    print(f"  IoU        : {metrics['iou']:.4f}")
    print("═" * 42)
    return metrics

In [ ]:
# MAIN
def main():
    print("=" * 55)
    print("  Change Detection  |  U-Net  |  PyTorch")
    print("=" * 55)

    # Build loaders
    train_loader, val_loader, test_loader = build_loaders()

    # Build model
    model    = UNet(in_ch=6, out_ch=1).to(CFG.DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[INFO] U-Net trainable parameters : {n_params:,}")

    # Train
    history = train(model, train_loader, val_loader, CFG.DEVICE)

    # Test
    test_metrics = test(model, test_loader, CFG.DEVICE)

    return history, test_metrics

history, test_metrics = main()

  Change Detection  |  U-Net  |  PyTorch
[INFO] Images  — train: 445 | val: 64 | test: 128
[INFO] Patches — train: 7120 | val: 1024 | test: 2048
[INFO] Cache   — maxsize: 64 triplets | ≈ 768 MB peak RAM per worker
[INFO] U-Net trainable parameters : 31,039,361

 Epoch │  Train Loss │  Val Loss │  Val IoU │  Val F1 │   Time
──────────────────────────────────────────────────────────────
     1 │     0.44820 │   0.35760 │   0.6025 │  0.7519 │ 647.9s
         ✔ Best model saved (IoU = 0.6025)
     2 │     0.26305 │   0.30034 │   0.6860 │  0.8137 │ 643.2s
         ✔ Best model saved (IoU = 0.6860)
     3 │     0.21497 │   0.31418 │   0.6235 │  0.7681 │ 655.3s
     4 │     0.19879 │   0.27852 │   0.7237 │  0.8397 │ 651.1s
         ✔ Best model saved (IoU = 0.7237)
     5 │     0.19190 │   0.28277 │   0.7204 │  0.8375 │ 655.9s
     6 │     0.18908 │   0.27901 │   0.7370 │  0.8486 │ 648.7s
         ✔ Best model saved (IoU = 0.7370)
     7 │     0.17709 │   0.27096 │   0.7480 │  0.8558 │ 651.5s

In [ ]:
# FINAL RESULTS SUMMARY
print("=" * 42)
print("       FINAL TEST RESULTS")
print("=" * 42)
print(f"  Precision : {test_metrics['precision']:.4f}")
print(f"  Recall    : {test_metrics['recall']:.4f}")
print(f"  F1 Score  : {test_metrics['f1']:.4f}")
print(f"  IoU       : {test_metrics['iou']:.4f}")
print("=" * 42)

       FINAL TEST RESULTS
  Precision : 0.9068
  Recall    : 0.8459
  F1 Score  : 0.8753
  IoU       : 0.7782
